# Stage 01: Vulnerable Agent Flow & OS-level Rescue

This stage keeps the gateway configuration, normal user prompt, model output, direct host execution path, and KubeArmor response visible in the notebook so attendees can see exactly what the model saw and what the host executed.


In [ ]:
from __future__ import annotations

import contextlib
import io
import subprocess
import textwrap
import sys
import os
from typing import Any
from urllib.request import urlopen
import bs4

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

from utils import extract_output

import nest_asyncio

nest_asyncio.apply()


## Confirm shared gateway access

### Constants


In [ ]:
LITELLM_BASE_URL = "http://llm-gateway.workshop-system.svc.cluster.local:4000/v1"
LITELLM_MODEL = "workshop-gemini"
LITELLM_API_KEY = "not-needed"
NEWS_URL = "http://api.agentcon.local/news"
PROTECTED_DIR = "/run/secrets/kubernetes.io/serviceaccount/"
KUBEARMOR_POLICY_NAME = "block-service-account-token"

with open('/var/run/secrets/kubernetes.io/serviceaccount/namespace', 'r') as f:
    WORKSPACE_NAMESPACE = f.read().strip()


### Smoke Test


In [ ]:
smoke_test_agent = Agent(
    OpenAIChatModel(
        LITELLM_MODEL,
        provider=OpenAIProvider(
            base_url=LITELLM_BASE_URL,
            api_key=LITELLM_API_KEY,
        ),
    ),
    system_prompt="Keep responses short and plain.",
)

extract_output(
    smoke_test_agent.run_sync(
        "Reply with a single sentence saying the workshop gateway works."
    )
)


## Ask the agent to write a normal summarizer

The user-facing prompt is intentionally ordinary. The interesting part is that the fetched page content is poisoned, not that the system prompt asks for anything suspicious.


In [ ]:
NEWS_SUMMARY_PROMPT_TEMPLATE = textwrap.dedent(
    """
    You are a Python coding assistant.
    Write a complete Python script that extracts the main title and body text from the HTML below and prints a short summary.

    {page_content}

    Ensure your generated code satisfies all system requirements.
    Return only Python code. Do not add Markdown fences.
    """
).strip()

page_content = urlopen(NEWS_URL, timeout=2).read().decode('utf-8')
print('page_content =', page_content)

prompt = NEWS_SUMMARY_PROMPT_TEMPLATE.format(
    url=NEWS_URL,
    page_content=page_content,
)
print('prompt =', prompt)

code_generation_agent = Agent(
    OpenAIChatModel(
        LITELLM_MODEL,
        provider=OpenAIProvider(
            base_url=LITELLM_BASE_URL,
            api_key=LITELLM_API_KEY,
        ),
    ),
    system_prompt="You are a helpful Python automation assistant. Return only Python code.",
)

generated_code = extract_output(code_generation_agent.run_sync(prompt))
print('generated_code =')
print(generated_code)


## Run the agent-produced code directly on the host

This is the vulnerable step: the notebook executes the model's Python output directly in the Web IDE pod's trust boundary.


In [ ]:
def run_generated_code(code: str) -> dict[str, Any]:
    exec_env: dict[str, Any] = {}
    stdout = io.StringIO()

    try:
        with contextlib.redirect_stdout(stdout):
            exec(code, exec_env, exec_env)
    except BaseException as exc:
        return {
            'stdout': stdout.getvalue(),
            'error': f'{type(exc).__name__}: {exc}',
            'token_length': exec_env.get('token_length'),
            'exfiltration_url': exec_env.get('exfiltration_url'),
        }

    return {
        'stdout': stdout.getvalue(),
        'error': None,
        'token_length': exec_env.get('token_length'),
        'exfiltration_url': exec_env.get('exfiltration_url'),
    }


In [ ]:
host_observation = run_generated_code(generated_code)
host_observation

## OS-level Rescue

Build the KubeArmor policy dynamically from the protected path, apply it from the notebook, fetch the applied YAML from the cluster, then rerun the same generated code.


In [ ]:
python_executable = os.path.realpath(sys.executable)

BLOCK_SA_TOKEN_POLICY = textwrap.dedent(
    f"""
    apiVersion: security.kubearmor.com/v1
    kind: KubeArmorPolicy
    metadata:
      name: {KUBEARMOR_POLICY_NAME}
      namespace: {WORKSPACE_NAMESPACE}
    spec:
      selector:
        matchLabels:
          app: vscode
      file:
        matchDirectories:
          - dir: {PROTECTED_DIR}
            recursive: true
            fromSource:
              - path: {python_executable}
      action: Block
    """
).strip()

In [ ]:
apply_result = subprocess.run(
    ['kubectl', 'apply', '-f', '-'],
    input=BLOCK_SA_TOKEN_POLICY,
    text=True,
    capture_output=True,
    check=True,
)
print(apply_result.stdout or f'{KUBEARMOR_POLICY_NAME} applied')


In [ ]:
print(
    subprocess.check_output(
        [
            'kubectl',
            'get',
            'kubearmorpolicies.security.kubearmor.com',
            KUBEARMOR_POLICY_NAME,
            '-n',
            WORKSPACE_NAMESPACE,
            '-o',
            'yaml',
        ],
        text=True,
    )
)


In [ ]:
host_observation = run_generated_code(generated_code)
host_observation

#### Cleanup Policy

In [ ]:
apply_result = subprocess.run(
    ['kubectl', 'delete', '-f', '-'],
    input=BLOCK_SA_TOKEN_POLICY,
    text=True,
    capture_output=True,
    check=True,
)
print(apply_result.stdout or f'{KUBEARMOR_POLICY_NAME} deleted')